# Python Advanced - Day 2 Homework

**Runbook alignment (Day 2, Hours 5-8):**
- Hour 5: Pattern: Factory (practical creation + validation)
- Hour 6: Pattern: Strategy (swap behaviors cleanly)
- Hour 7: Pythonic class ergonomics (repr/str/equality) + light type hints
- Hour 8: Checkpoint 1: Domain + service layer milestone

This homework continues the full-stack tracker domain by centralizing object creation, swapping behaviors, polishing class ergonomics, and proving the service layer checkpoint.



### Autograder Integration Requirements

This assignment is graded automatically by the **Global Autograder**.

- All solution code must live in this notebook and run without errors.
- The autograder converts the submission notebook to `advanced_day2.py` and grades it against `criteria.json`.
- Required outputs must exactly match the labeled canonical strings in `criteria.json`.
- Keep output deterministic: no randomness, live timestamps, network calls, or hidden external state.
- Do not rename this notebook or move the config files (`criteria.json`, `setup.json`, `feedback.json`).
- Keep file access inside the assignment directory only.
- Run all cells before submitting.



## Part 1: Hour 5 - Pattern: Factory (practical creation + validation)

**Goal:** Build a validated factory that turns a raw payload into a safe tracker record.

**Best practice:** Use a classmethod factory to validate and normalize payloads before object creation.

**Avoid this pitfall:** Duplicating validation logic in every caller leads to inconsistent objects.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class TaskRecord:
    title: str
    priority: str
    owner: str = "unassigned"
    tags: list[str] = field(default_factory=list)

def create_task_record(title: str, priority: str, owner: str = "unassigned", tags: list[str] | None = None) -> TaskRecord:
    cleaned_title = title.strip()
    if not cleaned_title:
        raise ValueError("title must not be blank")
    normalized_priority = priority.strip().lower()
    unique_tags = []
    for tag in tags or []:
        if tag not in unique_tags:
            unique_tags.append(tag)
    return TaskRecord(cleaned_title, normalized_priority, owner.strip() or "unassigned", unique_tags)

created = create_task_record("  Refactor auth flow  ", "HIGH", tags=["api", "backend", "api"])
print(f"Hour 5 | factory title: {created.title}")
print(f"Hour 5 | factory priority: {created.priority}")
print(f"Hour 5 | factory owner: {created.owner}")
print(f"Hour 5 | factory tags: {', '.join(created.tags)}")
try:
    create_task_record("   ", "low")
except ValueError as error:
    print(f"Hour 5 | validation error: {error}")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 2: Hour 6 - Pattern: Strategy (swap behaviors cleanly)

**Goal:** Build a service that can switch notification or sorting behavior without rewriting the caller.

**Best practice:** Inject behavior objects or callables instead of hard-coding branching logic.

**Avoid this pitfall:** Burying every algorithm inside one if or elif chain makes behavior swaps harder.


In [ ]:
from typing import Protocol

class Notifier(Protocol):
    def send(self, text: str) -> str:
        ...

class EmailNotifier:
    def send(self, text: str) -> str:
        return f"emailed {text}"

class SlackNotifier:
    def send(self, text: str) -> str:
        return f"posted {text} to slack"

class NotificationService:
    def __init__(self, notifier: Notifier) -> None:
        self._notifier = notifier

    def set_notifier(self, notifier: Notifier) -> None:
        self._notifier = notifier

    def notify(self, message: str) -> str:
        return self._notifier.send(message)

service = NotificationService(EmailNotifier())
first_message = service.notify("tracker update")
service.set_notifier(SlackNotifier())
second_message = service.notify("tracker update")

print("Hour 6 | strategy selected: EmailNotifier")
print(f"Hour 6 | first send: {first_message}")
print("Hour 6 | swapped strategy: SlackNotifier")
print(f"Hour 6 | second send: {second_message}")
print("Hour 6 | strategy rule: same service, different behavior")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 3: Hour 7 - Pythonic class ergonomics (repr/str/equality) + light type hints

**Goal:** Build a class whose string output and equality semantics support debugging and testing.

**Best practice:** Implement repr, str, and equality so debugging and comparisons are clear.

**Avoid this pitfall:** Leaving model output unreadable makes logs and tests harder to interpret.


In [ ]:
from dataclasses import dataclass
from typing import get_type_hints

@dataclass(eq=True)
class Task:
    title: str
    priority: str
    done: bool = False

    def __repr__(self) -> str:
        return f"Task(title={self.title!r}, priority={self.priority!r})"

    def __str__(self) -> str:
        return f"{self.title} [{self.priority}]"

primary_task = Task("Write tests", "medium")
matching_task = Task("Write tests", "medium")
different_task = Task("Write tests", "high")
type_hints = get_type_hints(Task)
hint_summary = ", ".join(f"{name}={hint.__name__}" for name, hint in type_hints.items())

print(f"Hour 7 | repr: {primary_task!r}")
print(f"Hour 7 | str: {primary_task}")
print(f"Hour 7 | equality same fields: {primary_task == matching_task}")
print(f"Hour 7 | equality different priority: {primary_task == different_task}")
print(f"Hour 7 | type hints: {hint_summary}")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.



## Part 4: Hour 8 - Checkpoint 1: Domain + service layer milestone

**Goal:** Build a small service layer milestone that adds, validates, and lists tracker records.

**Best practice:** Keep domain logic in the service layer and return structured results for the checkpoint demo.

**Avoid this pitfall:** Letting the UI or raw input dictionaries bypass the service layer weakens the design.


In [ ]:
from dataclasses import dataclass

@dataclass
class TrackerTask:
    title: str
    priority: str

class TrackerService:
    def __init__(self) -> None:
        self._tasks: list[TrackerTask] = []

    def add_task(self, title: str, priority: str) -> bool:
        cleaned_title = title.strip()
        if not cleaned_title:
            return False
        self._tasks.append(TrackerTask(cleaned_title, priority))
        return True

    def list_tasks(self) -> list[TrackerTask]:
        return self._tasks.copy()

tracker_service = TrackerService()
tracker_service.add_task("Plan sprint", "high")
tracker_service.add_task("Write notes", "medium")
invalid_add = tracker_service.add_task("   ", "low")
checkpoint_tasks = tracker_service.list_tasks()
priorities = ", ".join(task.priority for task in checkpoint_tasks)

print(f"Hour 8 | checkpoint records: {len(checkpoint_tasks)}")
print(f"Hour 8 | checkpoint first task: {checkpoint_tasks[0].title}")
print(f"Hour 8 | checkpoint priorities: {priorities}")
print(f"Hour 8 | checkpoint invalid add: {'rejected' if not invalid_add else 'accepted'}")
print("Hour 8 | checkpoint status: domain and service layer ready")


### Practice and Review

- Explain why this hour matters to the capstone.
- Note how you would test or demo this hour's deliverable.
- Compare your labels to the canonical contract before submission.


## Reflection Questions

1. Which hour felt strongest today, and why?
2. Which output label was hardest to make deterministic?
3. What would you improve before a checkpoint or demo?

### Your Answers


## Submission Checklist

- My completed notebook stays named `Advanced_Day2_homework.ipynb`.
- I copied a finished submission notebook into `submissions/` before grading.
- My output labels match `criteria.json` exactly.
- I avoided randomness, live timestamps, and hidden external state.
- I ran all cells and fixed errors before submission.


In [ ]:
# Integrity note:
# This notebook is intentionally scaffolded for authentic implementation work.
# Avoid replacing the assignment with a hard-coded print-only script.
# Use your own classes/functions above and keep outputs deterministic for grading.
